In [78]:
# -----------------------------------------------------------------------------
# [수정] 경로 재정의 (삭제된 Cell 1의 역할을 대신합니다)
# -----------------------------------------------------------------------------
from pathlib import Path

QP2_ROOT = Path(r"C:/QP2")
DATA_DIR = QP2_ROOT / "data"
META_DIR = DATA_DIR / "meta"      # <--- 이 줄이 없어서 에러가 난 거예요!
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
GDELT_DIR = RAW_DIR / "gdelt"

# 폴더가 없으면 만들어주는 센스
META_DIR.mkdir(parents=True, exist_ok=True)
GDELT_DIR.mkdir(parents=True, exist_ok=True)

In [79]:
# 컬럼 이름 확인용 코드
print(universe.columns.tolist())
print(universe.head(3))  # 데이터 상단 3줄 미리보기

['ticker_sp', 'ticker_yahoo', 'cik', 'security_name', 'GICS Sector', 'GICS Sub-Industry']
  ticker_sp ticker_yahoo         cik        security_name  GICS Sector  \
0       MMM          MMM  0000066740                   3M  Industrials   
1       AOS          AOS  0000091142          A. O. Smith  Industrials   
2       ABT          ABT  0000001800  Abbott Laboratories  Health Care   

          GICS Sub-Industry  
0  Industrial Conglomerates  
1         Building Products  
2     Health Care Equipment  


In [ ]:
import pandas as pd
import requests
import zipfile
import io
import time
from pathlib import Path
from tqdm import tqdm

# 1. 환경 설정 및 유니버스 로드 (이미 정의되어 있다면 생략 가능하지만, 안전을 위해 포함)
QP2_ROOT = Path(r"C:/QP2")
GDELT_DIR = QP2_ROOT / "data" / "raw" / "gdelt"
GDELT_DIR.mkdir(parents=True, exist_ok=True)

universe = pd.read_parquet(QP2_ROOT / "data" / "meta" / "sp500_universe.parquet")
all_tickers = set(universe['ticker_yahoo'].astype(str).str.upper().tolist())
all_names = set(universe['security_name'].astype(str).str.upper().tolist())
target_keywords = all_tickers | all_names
name_to_ticker = universe.set_index('security_name')['ticker_yahoo'].to_dict()

# 2. [수정 포인트] 수집하고 싶은 날짜 범위 설정 (2022년 ~ 2025년)
start_date = "2022-01-01"
end_date = "2025-12-31"  # 또는 원하는 종료 날짜
date_range = pd.date_range(start=start_date, end=end_date).strftime("%Y%m%d")

print(f"📅 총 {len(date_range)}일치 뉴스 트럭이 출발 대기 중입니다!")

# 3. 장기 수집 루프 가동
for target_date in tqdm(date_range, desc="전체 수집 진행도"):
    save_path = GDELT_DIR / f"gdelt_final_{target_date}.parquet"
    
    # 이미 받은 날짜는 건너뜁니다 (중복 수집 방지)
    if save_path.exists():
        continue
        
    url = f"http://data.gdeltproject.org/gkg/{target_date}.gkg.csv.zip"
    
    try:
        resp = requests.get(url, timeout=30)
        if resp.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
                with z.open(f"{target_date}.gkg.csv") as f:
                    day_df = pd.read_csv(f, sep='\t', encoding='utf-8', on_bad_lines='skip',
                                         usecols=[0, 6, 10], names=['date', 'organizations', 'url'],
                                         dtype={'date': str})

            # 초고속 필터링 로직
            day_df['org_list'] = day_df['organizations'].str.split(';')
            exploded = day_df.explode('org_list')
            # 'Apple,123' 형태에서 이름만 추출 후 대문자 변환
            exploded['org_name'] = exploded['org_list'].str.split(',').str[0].str.upper()
            
            # 우리 종목만 필터링
            filtered = exploded[exploded['org_name'].isin(target_keywords)].copy()
            filtered['ticker'] = filtered['org_name'].apply(lambda x: name_to_ticker.get(x, x))
            
            if not filtered.empty:
                final_df = filtered.drop_duplicates(subset=['url', 'ticker'])
                final_df.to_parquet(save_path, index=False)
        
        # 서버 부하 방지를 위해 아주 살짝만 쉬기
        time.sleep(0.1)
        
    except Exception as e:
        # 에러가 나도 멈추지 않고 다음 날짜로 넘어갑니다.
        print(f"⚠️ {target_date} 수집 중 건너뜀: {e}")
        continue

print(f"\n✅ 수집 완료! 데이터는 {GDELT_DIR} 폴더에 차곡차곡 쌓였습니다.")

📅 총 1461일치 뉴스 트럭이 출발 대기 중입니다!


전체 수집 진행도: 100%|██████████| 1461/1461 [1:49:24<00:00,  4.49s/it] 


✅ 수집 완료! 데이터는 C:\QP2\data\raw\gdelt 폴더에 차곡차곡 쌓였습니다.


: 